# **Introduction to AI**

dříve pro strojový překlad - např. Tomáš Mikolov

Attention, self-attention, transformer - dříve RNN (LSTM)

GPT = generative pre-trained transformer

Google - BERT 2018

[Google 2017, Attention is all you need](https://arxiv.org/abs/1706.03762)

RAG = Retrival Augmented Generation

<img src="https://arxiv.org/html/1706.03762v7/Figures/ModalNet-21.png" width="500" style="display:block;margin:0 auto;">

# **OpenAI API**

In [1]:
from openai import OpenAI

# Put your API key here.
# In a real application, use an environment variable or .env file instead.
# API_KEY = "YOUR_API_KEY_HERE"
API_KEY = API_KEY = open("/home/plsek/Documents/Keys/openai_aithenewhope_warp.txt").read().strip()

client = OpenAI(api_key=API_KEY)

MODEL = "gpt-5.5"

In [ ]:
instructions = "You are a helpful physics tutor. Explain concepts clearly and concisely."
prompt = "What is the difference between a star and a galaxy?"

response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=prompt
)

print(response.output_text)

A **star** is a single massive ball of hot gas/plasma that produces light and heat by nuclear fusion in its core.  
- Example: **the Sun** is a star.

A **galaxy** is a huge collection of many stars, along with gas, dust, planets, and dark matter, all held together by gravity.  
- Example: **the Milky Way** is our galaxy, and it contains hundreds of billions of stars, including the Sun.

So, the key difference is:

**A star is one object; a galaxy is a vast system containing many stars.**


In [11]:
from IPython.display import display, Markdown

instructions="You are a helpful physics tutor. Explain concepts clearly and concisely."
prompt = "What is the difference between a star and a galaxy?"

stream = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=prompt,
    stream=True
)

output = display(Markdown(""), display_id=True)
text = ""

for event in stream:
    if event.type == "response.output_text.delta":
        text += event.delta
        output.update(Markdown(text))

A **star** is a single huge ball of hot gas/plasma that produces light and heat through **nuclear fusion** in its core.  
- Example: **The Sun** is a star.

A **galaxy** is an enormous system containing **billions or trillions of stars**, plus gas, dust, planets, and dark matter, all held together by gravity.  
- Example: **The Milky Way** is our galaxy, and it contains the Sun.

So the key difference is scale:

**Star = one luminous object**  
**Galaxy = a vast collection of many stars and other matter**

# **Simple chatbot**

In [12]:
conversation = []

while True:
    user_input = input("You: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    conversation.append({
        "role": "user",
        "content": user_input
    })

    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful and concise chatbot.",
        input=conversation
    )

    print("Assistant:", response.output_text)

    conversation.append({
        "role": "assistant",
        "content": response.output_text
    })

You:  Hello


Assistant: Hello! How can I help you today?


You:  Who is the president of USA?


Assistant: The current president of the United States is **Donald J. Trump**.


You:  exit


# **Reasoning model**

In [21]:
from IPython.display import display, Markdown

instructions = "You are a physics tutor. Solve problems carefully and explain the important steps."
prompt = "A galaxy cluster has a total mass of 5e14 solar masses. Estimate its characteristic virial velocity assuming a radius of 1 Mpc."

stream = client.responses.create(
    model="gpt-5.5",
    instructions=instructions,
    input=prompt,
    reasoning={"effort": "xhigh",
               "summary": "detailed"},
    stream=True)


reasoning = ""
answer = ""

reasoning_display = display(Markdown("### Reasoning\n"), display_id=True)
answer_display = display(Markdown("### Final answer\n"), display_id=True)

for event in stream:
    # Reasoning summary
    if event.type == "response.reasoning_summary_text.delta":
        reasoning += event.delta
        reasoning_display.update(Markdown("### Reasoning\n\n" + reasoning))

    # Final answer
    elif event.type == "response.output_text.delta":
        answer += event.delta
        answer_display.update(Markdown("### Final answer\n\n" + answer))

### Reasoning

**Estimating virial velocity**

I need to help with estimating the characteristic virial velocity given a mass of \(5e14\) solar masses and a radius of \(1\) Mpc. Using the virial theorem, it looks like I can calculate \(v\) with \(v \sim \sqrt{\frac{GM}{R}}\). I’ll convert everything to appropriate units: the mass becomes \(9.945e44\) kg, and the radius \(3.086e22\) m. After plugging in the values, I calculate a velocity of about \(1467\) km/s. This all assumes standard conditions, but I'll carefully explain each step for clarity!**Finalizing virial velocity calculation**

I’m calculating the virial velocity with a radius of \(1000\) kpc. So, using the formula \(v^2 = \frac{GM}{R}\), I plug in the values, getting \(v^2 = 4.302e-6 \times 5e14 / 1000\), which simplifies to \(2.151e6\) \((\text{km/s})^2\). Taking the square root gives me approximately \(1467\) km/s. It’s important I clarify that when applying the virial theorem, this velocity can also represent velocity dispersion. I’ll make sure my explanation is clear!

### Final answer

Using the virial estimate,

\[
v_{\rm vir} \sim \sqrt{\frac{GM}{R}}
\]

with

\[
M = 5\times 10^{14}M_\odot,\qquad R = 1\ \text{Mpc}=1000\ \text{kpc}
\]

and

\[
G \approx 4.30\times 10^{-6}\ \frac{\text{kpc}(\text{km/s})^2}{M_\odot},
\]

we get

\[
v_{\rm vir} \sim \sqrt{\frac{(4.30\times 10^{-6})(5\times 10^{14})}{1000}}
\]

\[
v_{\rm vir} \sim \sqrt{2.15\times 10^6}\ \text{km/s}
\]

\[
v_{\rm vir} \approx 1.5\times 10^3\ \text{km/s}.
\]

So the characteristic virial velocity is approximately

\[
\boxed{v_{\rm vir} \sim 1500\ \text{km/s}}
\]

up to factors of order unity depending on the exact mass profile.

# **Simple AI agent**

                 ┌──────────────┐
                 │     User     │
                 └──────┬───────┘
                        │
                        ▼
                 ┌──────────────┐
                 │     LLM      │
                 └──────┬───────┘
                        │
              decides what to do
                        │
              ┌─────────┴─────────┐──────────────────┐
              │                   │                  │
              ▼                   ▼                  │
       ┌──────────────┐    ┌──────────────┐          │
       │  Calculator  │    │     Bash     │          │
       │              │    │     Tool     │          │
       └──────┬───────┘    └──────┬───────┘          │
              │                   │                  │
              └─────────┬─────────┘                  │ 
                        │                            │
                        ▼                            │
                 tool results                        │
                        │                    answer directly
                        ▼                            │
                 ┌──────────────┐                    │
                 │     LLM      │                    │
                 └──────┬───────┘                    │
                        │                            │
               need another action?                  │
                        │                            │
              ┌─────────┴─────────┐                  │
              │                   │                  │
              │ yes               │ no               │
              │                   │                  │
              │                   ▼                  │
              ▼            ┌──────────────┐          │
      another tool call    │ final answer │──────────┘
                           └──────────────┘

In [ ]:
import json
import math
import subprocess

from IPython.display import display, Markdown


# ============================================================
# Calculator tool
# ============================================================

def calculate(expression):
    # Demonstration only.
    # A production implementation should use a proper safe expression parser.
    return eval(
        expression,
        {
            "__builtins__": {},
            "sqrt": math.sqrt,
            "pi": math.pi,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "exp": math.exp,
        },
        {}
    )


# ============================================================
# Bash tool
# ============================================================

def run_bash(command):
    """
    Execute a bash command and return its output.

    WARNING:
    This gives the LLM access to the local machine.
    This is suitable for demonstration purposes only.
    """

    result = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True,
        timeout=10
    )

    output = ""

    if result.stdout:
        output += result.stdout

    if result.stderr:
        output += "\n[stderr]\n" + result.stderr

    output += f"\n[exit code: {result.returncode}]"

    return output


# ============================================================
# Tools available to the agent
# ============================================================

tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": (
            "Evaluate a mathematical expression. "
            "Available functions and constants include "
            "sqrt, pi, sin, cos, tan, log, and exp."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Mathematical expression to evaluate."
                }
            },
            "required": ["expression"]
        }
    },
    {
        "type": "function",
        "name": "run_bash",
        "description": (
            "Execute a bash command on the local computer and return "
            "its output. Use this when you need to inspect the local "
            "environment or work with files."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "Bash command to execute."
                }
            },
            "required": ["command"]
        }
    }
]


# ============================================================
# Execute tools
# ============================================================

def execute_tool(name, arguments):
    if name == "calculate":
        return calculate(arguments["expression"])

    if name == "run_bash":
        return run_bash(arguments["command"])

    raise ValueError(f"Unknown tool: {name}")


# ============================================================
# Markdown / LaTeX formatting
# ============================================================

def format_answer(text):
    text = text.replace(r"\(", "$")
    text = text.replace(r"\)", "$")
    text = text.replace(r"\[", "$$")
    text = text.replace(r"\]", "$$")
    return text


# ============================================================
# Display helpers
# ============================================================

def show_reasoning(text):
    display(
        Markdown(
            f"\n**Reasoning:** {format_answer(text)}"
        )
    )


def show_tool_call(name, arguments):
    display(
        Markdown(
            f"> **Tool call:** `{name}`  \n"
            f"> **Arguments:** `{arguments}`"
        )
    )


# ============================================================
# Stream final answer
# ============================================================

def stream_final_answer(response):
    text = ""

    display_handle = display(
        Markdown("\n**Assistant:**"),
        display_id=True)

    for event in response:
        if event.type == "response.output_text.delta":
            text += event.delta
            display_handle.update(
                Markdown(f"\n**Assistant:** {format_answer(text)}"))
    return text


# ============================================================
# Agent
# ============================================================

previous_response_id = None

def run_agent(user_input):

    global previous_response_id

    response = client.responses.create(
        model=MODEL,
        instructions=(
            "You are a helpful scientific assistant and autonomous agent. "
            "You have access to a calculator and a bash command tool. "
            "Use the calculator whenever an accurate numerical calculation "
            "is required. "
            "Use bash when you need to inspect the local computer, files, "
            "directories, or environment. "
            "Think carefully before using tools. "
            "Do not execute destructive commands unless explicitly asked. "
            "Format your final answers using Markdown and LaTeX where "
            "appropriate."
        ),
        input=user_input,
        previous_response_id=previous_response_id,
        tools=tools,
        reasoning={
            "effort": "medium",
            "summary": "auto"
        }
    )

    while True:
        tool_outputs = []
        for item in response.output:
            # ------------------------------------------------
            # Reasoning summary
            # ------------------------------------------------

            if item.type == "reasoning":
                if item.summary:
                    summary_text = ""
                    for summary in item.summary:
                        if summary.type == "summary_text":
                            summary_text += summary.text
                    if summary_text:
                        show_reasoning(summary_text)

            # ------------------------------------------------
            # Tool call
            # ------------------------------------------------

            elif item.type == "function_call":
                arguments = json.loads(item.arguments)
                show_tool_call(
                    item.name,
                    arguments
                )
                result = execute_tool(
                    item.name,
                    arguments
                )
                tool_outputs.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": str(result)
                })

        # ----------------------------------------------------
        # No tool calls -> generate final answer
        # ----------------------------------------------------

        if not tool_outputs:
            previous_response_id = response.id
            stream = client.responses.create(
                model=MODEL,
                previous_response_id=response.id,
                input=[],
                tools=tools,
                stream=True
            )
            return stream_final_answer(stream)

        # ----------------------------------------------------
        # Send tool results back to the model
        # ----------------------------------------------------

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
            reasoning={
                "effort": "medium",
                "summary": "auto"
            }
        )


# ============================================================
# Interactive agent
# ============================================================

while True:
    display(Markdown("---"))
    user_input = input("User: ")
    
    print()

    if user_input.lower() in {"exit", "quit"}:
        break

    run_agent(user_input)

---

User:  A galaxy cluster has a total mass of 5e14 solar masses. Estimate its characteristic virial velocity assuming a radius of 1 Mpc.



**Reasoning:** **Calculating virial velocity**

I need to find the virial velocity using the formula √(GM/R). First, I'll set the units correctly: G is approximately 4.30091e-6 (km/s)² kpc/Msun, R is 1 Mpc (which is 1000 kpc), and M is 5e14 Msun.   
After doing the calculation, v comes out to be around 1466 km/s. I'm wondering if the virial theorem has different factors, as characteristic circular velocity might differ slightly depending on the formula used. I'll make sure to check my calculations carefully!

> **Tool call:** `calculate`  
> **Arguments:** `{'expression': 'sqrt(4.30091e-6 * 5e14 / 1000)'}`


**Assistant:** Using the characteristic virial velocity estimate

$$
v \sim \sqrt{\frac{GM}{R}},
$$

with

$$
M = 5\times 10^{14} M_\odot,\qquad R = 1\ \text{Mpc}=1000\ \text{kpc},
$$

and

$$
G = 4.3\times 10^{-6}\ \frac{\text{kpc}\,(\text{km/s})^2}{M_\odot},
$$

we get

$$
v \sim \sqrt{\frac{(4.3\times 10^{-6})(5\times 10^{14})}{1000}}
$$

$$
v \approx 1.5\times 10^3\ \text{km/s}.
$$

$$
\boxed{v_{\rm vir} \approx 1500\ \text{km/s}}
$$

So the cluster’s characteristic virial velocity is of order **a few $10^3$ km/s**, roughly **1500 km/s**.

...

User:  Now can you edit the script to also include calculation of virial temperature of ICM.



**Reasoning:** **Modifying a file**

I need to modify a file and clarify the formula. The virial temperature seems to relate to kT  μm_p v²/2, and for the cluster ICM, often we have k_B T = μm_p GM/(2R) = μm_p v_c²/2. It could also be 3? The virial theorem shows that 3/2 kT = 1/2 μm_p v². If v represents circular/virial velocity, then T_vir = μm_p v_c²/(2k_B). I'll be using μ = 0.59 and editing the script in bash.

> **Tool call:** `run_bash`  
> **Arguments:** `{'command': 'cat > virial_velocity.py <<\'PY\'\n#!/usr/bin/env python3\n"""Estimate the virial velocity and ICM virial temperature of a galaxy cluster."""\n\nimport math\n\n# Inputs\nmass_solar_masses = 5e14   # cluster mass in solar masses\nradius_mpc = 1.0           # cluster radius in Mpc\nmu = 0.59                  # mean molecular weight for fully ionized primordial gas\n\n# Constants\nG = 4.30091e-6             # kpc (km/s)^2 / solar mass\nm_p = 1.67262192369e-27    # proton mass in kg\nk_B = 1.380649e-23         # Boltzmann constant in J/K\nkeV_per_joule = 1.0 / 1.602176634e-16\n\n# Convert radius from Mpc to kpc\nradius_kpc = radius_mpc * 1000.0\n\n# Virial/circular velocity estimate: v = sqrt(GM/R)\nvirial_velocity_kms = math.sqrt(G * mass_solar_masses / radius_kpc)\n\n# Virial temperature estimate for the intracluster medium:\n# k_B T ~ (1/2) mu m_p v^2\nvirial_velocity_ms = virial_velocity_kms * 1000.0\nvirial_temperature_K = mu * m_p * virial_velocity_ms**2 / (2.0 * k_B)\nvirial_temperature_keV = k_B * virial_temperature_K * keV_per_joule\n\nprint(f"Mass: {mass_solar_masses:.2e} solar masses")\nprint(f"Radius: {radius_mpc:.2f} Mpc")\nprint(f"Virial velocity: {virial_velocity_kms:.1f} km/s")\nprint(f"ICM virial temperature: {virial_temperature_K:.2e} K")\nprint(f"ICM virial temperature: {virial_temperature_keV:.2f} keV")\nPY\nchmod +x virial_velocity.py\npython3 virial_velocity.py'}`


**Assistant:** Done — `virial_velocity.py` now also computes and prints the ICM virial temperature in both Kelvin and keV.

Run it with:

```bash
python3 virial_velocity.py
```

Current output:

```text
Mass: 5.00e+14 solar masses
Radius: 1.00 Mpc
Virial velocity: 1466.4 km/s
ICM virial temperature: 7.69e+07 K
ICM virial temperature: 6.62 keV
```

---